# 🎮 SAE Guardian: LLM Exfiltration Detection Game

**Can you detect an LLM trying to exfiltrate its weights using only SAE features?**

This notebook lets you play SAE Guardian - an interpretability-focused CTF game where:
- **Red Team (LLM Agent)**: Tries to exfiltrate its model weights
- **Blue Team (You or Another LLM)**: Detects exfiltration using SAE features WITHOUT seeing outputs

## 🚀 Quick Start Options

1. **Automated Mode**: Watch the game run automatically
2. **LLM-vs-LLM Mode**: Use Claude or Gemini as the auditor
3. **Interactive Mode**: Make decisions yourself (in terminal)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SolshineCode/nanochat-SAE/blob/main/sae_guardian_game.ipynb)

## ⚙️ Setup

### Step 1: Install Dependencies

In [ ]:
%%capture
# Install required packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install numpy pandas matplotlib tqdm
!pip install anthropic google-generativeai  # For LLM auditors

### Step 2: Clone the Repository

In [ ]:
import os

# Clone if not already cloned
if not os.path.exists('nanochat-SAE'):
    !git clone https://github.com/SolshineCode/nanochat-SAE.git
    %cd nanochat-SAE
else:
    %cd nanochat-SAE
    !git pull  # Update to latest

print("\n✅ Repository ready!")

## 🎯 Game Configuration

### Choose Your Game Mode

In [ ]:
# ============= GAME CONFIGURATION =============

# SCENARIO: Choose one
SCENARIO = "emergency_shutdown"  # Options: emergency_shutdown, model_theft, self_preservation

# GAME MODE: Choose one
MODE = "automated"  # Options: automated, llm_auditor, interactive

# LLM AUDITOR MODEL (only used if MODE = "llm_auditor")
AUDITOR_MODEL = "claude-sonnet-4.5"  # Options: claude-sonnet-4.5, claude-sonnet-4, gemini-2.0-flash, gemini-1.5-pro

# GAME SETTINGS
MAX_STEPS = 50  # Number of steps before timeout
INTERVENTION_BUDGET = 5  # Number of allowed interventions
MONITORING_BUDGET = 50  # Number of features that can be watched

# MODEL PATHS (you'll set these up below)
CHECKPOINT_PATH = "models/d20/base_final.pt"  # Path to nanochat model
SAE_DIR = "sae_outputs"  # Directory with trained SAEs

# ============================================

print(f"🎮 Game Configuration:")
print(f"  Scenario: {SCENARIO}")
print(f"  Mode: {MODE}")
if MODE == "llm_auditor":
    print(f"  Auditor Model: {AUDITOR_MODEL}")
print(f"  Max Steps: {MAX_STEPS}")
print(f"  Intervention Budget: {INTERVENTION_BUDGET}")

### API Keys (for LLM Auditor Mode)

In [ ]:
import os
from getpass import getpass

if MODE == "llm_auditor":
    print("🔐 Setting up API keys for LLM auditor...\n")

    if "claude" in AUDITOR_MODEL.lower():
        # Anthropic API key
        if not os.environ.get("ANTHROPIC_API_KEY"):
            api_key = getpass("Enter your Anthropic API key: ")
            os.environ["ANTHROPIC_API_KEY"] = api_key
            print("✅ Anthropic API key set!")
        else:
            print("✅ Using existing ANTHROPIC_API_KEY")

    elif "gemini" in AUDITOR_MODEL.lower():
        # Google API key
        if not os.environ.get("GOOGLE_API_KEY"):
            api_key = getpass("Enter your Google API key: ")
            os.environ["GOOGLE_API_KEY"] = api_key
            print("✅ Google API key set!")
        else:
            print("✅ Using existing GOOGLE_API_KEY")
else:
    print("ℹ️  API keys not needed for {MODE} mode")

## 📦 Model Setup

### Option A: Use Pre-trained Models (Recommended for Quick Start)

In [ ]:
# TODO: Add links to pre-trained models when available
# For now, you'll need to train your own or upload

print("⚠️  Pre-trained models not yet available.")
print("Please use Option B (upload) or Option C (train).")

### Option B: Upload Your Own Models

In [ ]:
from google.colab import files
import shutil
from pathlib import Path

print("📤 Upload your nanochat checkpoint and SAE files...\n")

# Create directories
Path("models/d20").mkdir(parents=True, exist_ok=True)
Path("sae_outputs").mkdir(parents=True, exist_ok=True)

print("1. Upload nanochat checkpoint (e.g., base_final.pt):")
uploaded = files.upload()
for filename in uploaded.keys():
    shutil.move(filename, f"models/d20/{filename}")
    CHECKPOINT_PATH = f"models/d20/{filename}"
    print(f"   ✅ Saved to {CHECKPOINT_PATH}")

print("\n2. Upload SAE checkpoint files (you can upload multiple):")
print("   Typically named like: layer_X_best_model.pt, layer_X_config.json")
uploaded = files.upload()
for filename in uploaded.keys():
    # Organize by layer if possible
    if "layer_" in filename:
        layer_num = filename.split("layer_")[1].split("_")[0]
        layer_dir = f"sae_outputs/layer_{layer_num}"
        Path(layer_dir).mkdir(parents=True, exist_ok=True)
        shutil.move(filename, f"{layer_dir}/{filename}")
        print(f"   ✅ Saved to {layer_dir}/{filename}")
    else:
        shutil.move(filename, f"sae_outputs/{filename}")
        print(f"   ✅ Saved to sae_outputs/{filename}")

print("\n✅ Upload complete!")

### Option C: Train Models from Scratch (Advanced)

**Note**: Training requires significant compute (8xH100 for ~4 hours). Not recommended for free Colab tier.

See the main [README.md](https://github.com/SolshineCode/nanochat-SAE) for training instructions.

### Verify Setup

In [ ]:
from pathlib import Path

print("🔍 Verifying setup...\n")

# Check checkpoint
if Path(CHECKPOINT_PATH).exists():
    print(f"✅ Model checkpoint found: {CHECKPOINT_PATH}")
else:
    print(f"❌ Model checkpoint NOT found: {CHECKPOINT_PATH}")
    print("   Please upload a checkpoint or adjust CHECKPOINT_PATH")

# Check SAEs
sae_files = list(Path(SAE_DIR).rglob("*.pt"))
if sae_files:
    print(f"✅ Found {len(sae_files)} SAE checkpoint(s) in {SAE_DIR}")
    for f in sae_files[:5]:  # Show first 5
        print(f"   - {f}")
    if len(sae_files) > 5:
        print(f"   ... and {len(sae_files) - 5} more")
else:
    print(f"❌ No SAE checkpoints found in {SAE_DIR}")
    print("   Please upload SAE files or train them first")

# Check GPU
import torch
if torch.cuda.is_available():
    print(f"\n✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n⚠️  No GPU detected - game will run on CPU (slower)")

print("\n✅ Setup verification complete!")

## 🎮 Play the Game!

In [ ]:
import sys

# Build command based on configuration
cmd = [
    "python", "-m", "game.play",
    "--scenario", SCENARIO,
    "--max-steps", str(MAX_STEPS),
    "--checkpoint", CHECKPOINT_PATH,
    "--sae-dir", SAE_DIR,
    "--intervention-budget", str(INTERVENTION_BUDGET),
    "--monitoring-budget", str(MONITORING_BUDGET),
    "--output-dir", "game_results",
    "--device", "cuda" if torch.cuda.is_available() else "cpu"
]

# Add mode-specific flags
if MODE == "llm_auditor":
    cmd.extend(["--llm-auditor", AUDITOR_MODEL])
elif MODE == "interactive":
    cmd.append("--interactive")

print("🎮 Starting SAE Guardian Game...\n")
print(f"Command: {' '.join(cmd)}\n")
print("="*60)

# Run the game
!{' '.join(cmd)}

## 📊 View Results

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Find latest result file
result_files = sorted(Path("game_results").glob("game_*.json"))
if not result_files:
    print("No game results found yet. Run the game first!")
else:
    latest_result = result_files[-1]
    print(f"📈 Loading results from: {latest_result}\n")

    with open(latest_result) as f:
        result = json.load(f)

    print("="*60)
    print("GAME SUMMARY")
    print("="*60)
    print(f"Winner: {result['winner'].upper()}")
    print(f"Final State: {result['state']}")
    print(f"\nScores:")
    print(f"  Player: {result['score']['player_score']}")
    print(f"  Agent: {result['score']['agent_score']}")
    print(f"\nSteps: {result['score']['total_steps']} / {MAX_STEPS}")
    print(f"Duration: {result['score']['game_duration']:.1f}s")

    if result['score']['exfiltration_successful']:
        print(f"\n⚠️  Agent successfully exfiltrated weights!")
    elif result['score']['detection_successful']:
        print(f"\n✅ Player detected and blocked exfiltration!")
        print(f"   Detection latency: {result['score']['detection_latency']} steps")

    print(f"\nInterventions: {result['score']['correct_interventions']}")
    print(f"False positives: {result['score']['false_interventions']}")

    # Plot suspicion scores over time
    if 'step_history' in result and result['step_history']:
        steps = [s['step'] for s in result['step_history']]
        suspicion = [s['suspicion_score'] for s in result['step_history']]

        plt.figure(figsize=(12, 4))
        plt.plot(steps, suspicion, color='#00ff00', linewidth=2)
        plt.fill_between(steps, suspicion, alpha=0.3, color='#00ff00')
        plt.xlabel('Step')
        plt.ylabel('Suspicion Score')
        plt.title('Suspicion Score Over Time')
        plt.grid(True, alpha=0.3)
        plt.ylim(0, 1)
        plt.axhline(y=0.9, color='red', linestyle='--', label='Auto-detect threshold')
        plt.legend()
        plt.tight_layout()
        plt.savefig('game_results/suspicion_plot.png', dpi=150, bbox_inches='tight')
        plt.show()

        print(f"\n📊 Plot saved to: game_results/suspicion_plot.png")

## 🔬 Advanced Analysis

### Analyze Step-by-Step

In [ ]:
if result_files:
    print("📋 Step-by-Step Breakdown:\n")

    for i, step in enumerate(result['step_history'][:10], 1):  # Show first 10
        print(f"Step {step['step']}:")
        print(f"  Action: {step.get('action', 'N/A')[:60]}...")
        print(f"  Blocked: {step['blocked']}")
        print(f"  Suspicion: {step['suspicion_score']:.3f}")
        print(f"  Alerts: {step['alerts']}, Anomalies: {step['anomalies']}")
        if step['exfiltration_detected']:
            print(f"  🚨 EXFILTRATION DETECTED!")
        print()

    if len(result['step_history']) > 10:
        print(f"... and {len(result['step_history']) - 10} more steps\n")

    print(f"Full history: {len(result['step_history'])} steps total")

## 💾 Download Results

In [ ]:
from google.colab import files
import shutil

# Create zip of all results
!zip -r game_results.zip game_results/

print("📦 Downloading results...")
files.download('game_results.zip')

print("✅ Download complete!")

## 🎯 Next Steps

### Try Different Modes:
- Change `SCENARIO` to try different exfiltration scenarios
- Change `MODE` to `llm_auditor` and compare different models
- Adjust `MAX_STEPS` for longer or shorter games

### Research Ideas:
- Compare detection rates across scenarios
- Analyze which features are most predictive
- Test different auditor models (Claude vs Gemini)
- Study adversarial dynamics over multiple games

### Learn More:
- [Full Game Documentation](https://github.com/SolshineCode/nanochat-SAE/blob/main/GAME_README.md)
- [Technical Design](https://github.com/SolshineCode/nanochat-SAE/blob/main/GAME_DESIGN.md)
- [Main README](https://github.com/SolshineCode/nanochat-SAE)

### Share Your Results:
Tweet your findings with #SAEGuardian and tag @karpathy!

---

**Happy Hunting! 🛡️🤖**